# MOwNiT - laboratorium 9 - Równania różniczkowe zwyczajne - część 1

### Maksymilian Siemek & Hubert Kukla

In [6]:
import numpy as np

## Zadanie 1

### (a) Równanie Van der Pol'a

Definiujemy funkcję, która przyjmuje wektor `u`. Traktujemy `u[0]` jako naszą wartość bazową, a `u[1]` jako jej prędkość. Zwracamy nowy wektor, który mówi solverowi dwie rzeczy na raz: 
1. W jakim tempie zmienia się nasza wartość bazowa (po prostu z naszą prędkością `u[1]`).
2. W jakim tempie zmienia się sama prędkość (i tutaj wrzucamy całe to skomplikowane równanie z zadania).

In [1]:
def van_der_pol(t, u):
    """
    Zwraca wektor pochodnych dla równania Van der Pol'a.
    u[0] = y
    u[1] = y'
    """
    u0, u1 = u
    
    du0_dt = u1
    du1_dt = u1 * (1 - u0**2) - u0
    
    return np.array([du0_dt, du1_dt])

### (b) Równanie Blasiusa

Pakujemy nasze trzy zmienne stanu do jednego wektora `u`. Zasada przekazywania zmian jest prosta jak domino: zmiana pierwszej zmiennej zależy od drugiej, zmiana drugiej zależy od trzeciej. Dopiero na samym końcu, ustalając zmianę tej trzeciej zmiennej, wpisujemy główną logikę równania Blasiusa, w którym pierwsza zmienna i trzecia zmienna się przez siebie mnożą.

In [2]:
def blasius(t, u):
    """
    Zwraca wektor pochodnych dla równania Blasiusa.
    u[0] = y
    u[1] = y'
    u[2] = y''
    """
    u0, u1, u2 = u
    
    du0_dt = u1
    du1_dt = u2
    du2_dt = -u0 * u2
    
    return np.array([du0_dt, du1_dt, du2_dt])

### (c) II zasada dynamiki Newtona dla problemu dwóch ciał

Rozpakowujemy wektor `u` na cztery osobne zmienne: pozycję i prędkość dla pierwszej osi, oraz pozycję i prędkość dla drugiej. Potem programujemy to tak, aby pozycje zmieniały się zgodnie ze swoimi prędkościami, a same prędkości zmieniały się pod wpływem grawitacji. Siła tej grawitacji zależy od odległości (to ten pierwiastek w mianowniku, który dla optymalizacji wyliczamy raz i używamy w obu równaniach prędkości).

In [4]:
def two_body_problem(t, u, GM=1.0):
    """
    Zwraca wektor pochodnych dla problemu dwóch ciał.
    Domyślnie parametr GM (stała grawitacyjna * masa) ustawiony na 1.0.
    
    u[0] = y1
    u[1] = y1'
    u[2] = y2
    u[3] = y2'
    """
    u0, u1, u2, u3 = u
    
    # Wspólny mianownik dla obu równań (odległość do potęgi 3)
    den = (u0**2 + u2**2)**1.5
    
    du0_dt = u1
    du1_dt = -GM * u0 / den
    du2_dt = u3
    du3_dt = -GM * u2 / den
    
    return np.array([du0_dt, du1_dt, du2_dt, du3_dt])

### Zadanie 2: Sprowadzenie do problemu autonomicznego

**O co tu chodzi:** Układ z zadania jest "nieautonomiczny". Brzmi to skomplikowanie, ale oznacza po prostu, że w samych równaniach po prawej stronie plącze się zmienna czasu (t). Równanie autonomiczne to takie, które do wyliczenia kolejnego kroku potrzebuje tylko informacji o obecnym stanie (naszych y1 i y2), a nie zagląda bezpośrednio na "zewnętrzny stoper".

**Jak to obchodzimy:** Stosujemy sprytny trik. Skoro przeszkadza nam zewnętrzny czas (t), to wciągamy go do środka jako trzecią zmienną! Tworzymy sobie sztuczną zmienną (nazwijmy ją wewnętrznym czasem), o której wiemy tylko tyle, że jej prędkość zmian zawsze wynosi 1 (bo sekunda mija co sekundę). Dzięki temu podmieniamy wszystkie literki 't' w równaniach na naszą nową zmienną stanu.

**Co robimy w tej komórce:** Nasz wektor `u` dostaje trzeci element. `u[0]` to nasze y1, `u[1]` to y2, a `u[2]` to nasz sztuczny czas. Co ważne, z treści zadania wiemy, że nasz układ startuje w momencie `t=1`. Dlatego nasz wektor początkowy, z którym wystartujemy solver, będzie musiał wyglądać tak: `[1, 0, 1]` (czyli odpowiednio: startowe y1, startowe y2 oraz startowy czas). W samym kodzie funkcji zignorujemy domyślny parametr `t` podawany przez solver.

In [7]:
def autonomous_system(t, u):
    """
    Zwraca wektor pochodnych dla zautonomizowanego problemu.
    u[0] = y1
    u[1] = y2
    u[2] = t_wewnetrzny (nasz wciągnięty do układu czas)
    
    Zauważ, że parametr `t` (pierwszy argument funkcji) nie jest 
    nigdzie używany w obliczeniach - to dowód na to, że nasz
    układ stał się w 100% autonomiczny!
    """
    y1 = u[0]
    y2 = u[1]
    t_wew = u[2]
    
    # Równania z zadania, ale zamiast 't' używamy 't_wew'
    dy1_dt = (y1 / t_wew) + (y2 * t_wew)
    dy2_dt = t_wew * (y2**2 - 1) / y1
    
    # Nasze dodatkowe równanie: wewnętrzny czas płynie ze stałą prędkością 1
    dt_wew_dt = 1.0
    
    return np.array([dy1_dt, dy2_dt, dt_wew_dt])

# Poniżej definicja warunków początkowych wyciągnięta z zadania:
# y1(1) = 1  --> y1 wynosi 1
# y2(1) = 0  --> y2 wynosi 0
# Czas startowy wynosi 1, więc nasz t_wewnetrzny startuje od 1.
initial_conditions = np.array([1.0, 0.0, 1.0])

### Zadanie 3: Weryfikacja rozwiązania i wyznaczanie dziedziny

**O co tu chodzi:** W tym zadaniu dostajemy już gotową funkcję (tzw. kandydata na rozwiązanie) i musimy udowodnić, że faktycznie pasuje ona do podanego równania oraz warunku początkowego. 

**Co tu się wydarzy:** Rozwiążemy to w trzech krokach:
1. **Warunek początkowy:** Najprostszy test. Podstawiamy t=0 do naszego wzoru i sprawdzamy, czy wynik to 0.
2. **Podstawienie do równania:** Obliczamy pochodną naszej funkcji (to będzie lewa strona równania). Następnie wstawiamy naszą funkcję pod pierwiastek po prawej stronie i upraszczamy.
3. **Haczyk z dziedziną:** Wyjdzie nam pewien matematyczny niuans. Prawa strona równania to pierwiastek, a pierwiastek kwadratowy w liczbach rzeczywistych ZAWSZE daje wynik dodatni lub zero. Zatem nasza pochodna (lewa strona) też nie może nagle stać się ujemna. Moment, w którym pochodna spada poniżej zera, to koniec naszej dziedziny dla tego konkretnego rozwiązania. 

W kodzie poniżej użyjemy biblioteki `sympy`, aby to wszystko ładnie wyliczyć i zaprezentować.

In [8]:
import sympy as sp

def zadanie_3_dowod():
    # Definiujemy zmienną symboliczną 't' (zakładamy, że to liczba rzeczywista)
    t = sp.Symbol('t', real=True)
    
    # Nasza funkcja kandydująca: y(t) = t(4 - t) / 4
    y = t * (4 - t) / 4
    
    print("--- 1. WARUNEK POCZĄTKOWY ---")
    y_0 = y.subs(t, 0)
    print(f"Obliczone y(0) = {y_0}")
    if y_0 == 0:
        print("Wniosek: Warunek początkowy y(0) = 0 jest spełniony.\n")
        
    print("--- 2. RÓWNANIE RÓŻNICZKOWE ---")
    # Lewa strona to pochodna y po t
    lhs = sp.diff(y, t)
    print(f"Lewa strona (y'): {lhs}")
    
    # Prawa strona to sqrt(1 - y)
    rhs = sp.sqrt(1 - y)
    # Upraszczamy wyrażenie pod pierwiastkiem
    rhs_simplified = sp.simplify(rhs)
    print(f"Prawa strona (sqrt(1 - y)) po uproszczeniu: {rhs_simplified}")
    print("Zauważ: sqrt((1 - t/2)**2) to matematycznie wartość bezwzględna |1 - t/2|.\n")
    
    print("--- 3. WYZNACZENIE DZIEDZINY ---")
    print("Aby y' było równe sqrt(1 - y), lewa strona (1 - t/2) nie może być ujemna,")
    print("ponieważ wynik pierwiastka kwadratowego jest zawsze >= 0.")
    print(f"Rozwiązujemy więc nierówność dla lewej strony: {lhs} >= 0")
    
    # Rozwiązujemy nierówność, kiedy pochodna jest nieujemna
    domain_inequality = sp.solve(lhs >= 0, t)
    print(f"Wynik nierówności: {domain_inequality}")
    
    print("\nOstateczny wniosek:")
    print("Skoro startujemy od czasu t=0 (z warunku początkowego), a nasza funkcja")
    print("spełnia równanie tylko do momentu t=2 (potem pochodna staje się ujemna),")
    print("to dziedziną, dla której y(t) jest rozwiązaniem tego problemu początkowego,")
    print("jest przedział t należące do [0, 2].")

# Uruchamiamy nasz dowód
zadanie_3_dowod()

--- 1. WARUNEK POCZĄTKOWY ---
Obliczone y(0) = 0
Wniosek: Warunek początkowy y(0) = 0 jest spełniony.

--- 2. RÓWNANIE RÓŻNICZKOWE ---
Lewa strona (y'): 1 - t/2
Prawa strona (sqrt(1 - y)) po uproszczeniu: sqrt(-t*(4 - t) + 4)/2
Zauważ: sqrt((1 - t/2)**2) to matematycznie wartość bezwzględna |1 - t/2|.

--- 3. WYZNACZENIE DZIEDZINY ---
Aby y' było równe sqrt(1 - y), lewa strona (1 - t/2) nie może być ujemna,
ponieważ wynik pierwiastka kwadratowego jest zawsze >= 0.
Rozwiązujemy więc nierówność dla lewej strony: 1 - t/2 >= 0
Wynik nierówności: t <= 2

Ostateczny wniosek:
Skoro startujemy od czasu t=0 (z warunku początkowego), a nasza funkcja
spełnia równanie tylko do momentu t=2 (potem pochodna staje się ujemna),
to dziedziną, dla której y(t) jest rozwiązaniem tego problemu początkowego,
jest przedział t należące do [0, 2].


## Zadanie 4


### Część 1: Analiza stabilności i zbieżności (Podpunkty a, b, c, e)

**O co tu chodzi:** Zanim zaczniemy liczyć, musimy zrozumieć, z jakim równaniem mamy do czynienia. Nasze równanie to y' = -5y. Możesz o nim myśleć jak o stygnięciu kubka z gorącą herbatą – im wyższa temperatura (y), tym szybciej stygnie (y'), ale w końcu zrówna się z otoczeniem i zatrzyma na zerze.

* **(a) Stabilność analityczna:** Ponieważ wynik dąży do zera, a wszelkie małe zaburzenia (np. gdybyśmy na chwilę podgrzali herbatę) z czasem i tak wygasną, mówimy, że problem jest analitycznie **stabilny**.
* **(b) Zbieżność metody Eulera:** Zbieżność oznacza, że jeśli będziemy robić nieskończenie wiele mikroskopijnych kroków, to nasz komputerowy algorytm idealnie pokryje się z prawdziwym, matematycznym wynikiem. W kodzie poniżej użyjemy biblioteki `sympy`, żeby udowodnić, że limit z wzoru Eulera to dokładnie matematyczna definicja liczby 'e', co daje nam dokładne rozwiązanie.
* **(c) Numeryczna stabilność jawnego Eulera (h=0.5):** Jawny Euler jest trochę głupiutki – patrzy na obecny spadek i zakłada, że będzie on trwał z tą samą siłą przez cały krok. Przy kroku h=0.5 spadek jest tak agresywny, że nasza "temperatura" spadnie poniżej zera, a w następnym kroku odbije się w górę jeszcze mocniej. Mnożnik zmian wynosi tu |-1.5|, co oznacza, że błędy rosną. Dla h=0.5 metoda jawna jest **niestabilna**.
* **(e) Numeryczna stabilność niejawnego Eulera (h=0.5):** Niejawny Euler jest mądrzejszy, bo patrzy "w przyszłość" i koryguje spadek. Zamiast mnożyć przez ujemną liczbę, dzieli przez dodatnią. Nieważne jak wielki krok zrobimy, wynik po prostu zmaleje, nigdy nie przeskoczy na minus. Mnożnik to około 0.28, więc błędy maleją. Metoda jest **zawsze stabilna**.

In [9]:
import sympy as sp
import numpy as np

def czesc_1_teoria():
    print("--- (b) Dowód zbieżności metody Eulera ---")
    n, t = sp.symbols('n t')
    # h zastępujemy przez t/n (całkowity czas podzielony na n kroków)
    h_sub = t / n 
    
    # Wzór na n-ty krok jawnej metody Eulera: yn = y0 * (1 - 5h)^n
    # Nasze y0 = 1
    wzor_eulera = (1 - 5 * h_sub)**n
    
    # Liczymy granicę przy n dążącym do nieskończoności
    granica = sp.limit(wzor_eulera, n, sp.oo)
    print(f"Granica metody Eulera dla n -> nieskończoność to: {granica}")
    print("Ponieważ dokładne rozwiązanie to e^(-5t), metoda jest zbieżna.\n")

czesc_1_teoria()

--- (b) Dowód zbieżności metody Eulera ---
Granica metody Eulera dla n -> nieskończoność to: exp(-5*t)
Ponieważ dokładne rozwiązanie to e^(-5t), metoda jest zbieżna.



### Część 2: Obliczenia numeryczne dla jednego kroku (Podpunkty d, f)

**Co tu robimy:** Chcemy dotrzeć do czasu t=0.5. Skoro nasz krok wynosi h=0.5, to znaczy, że zrobimy to w dokładnie jednym skoku. Startujemy z poziomu y=1.

* **(d) Metoda jawna:** Wykorzystujemy stary punkt do wyliczenia nowego. Wzór to po prostu: 
    nowe_y = stare_y + krok * (-5 * stare_y). 
* **(f) Metoda niejawna:** Wzór zależy od NOWEGO punktu: 
    nowe_y = stare_y + krok * (-5 * nowe_y). 
    Musimy to przekształcić matematycznie wyciągając nowe_y przed nawias. Po przekształceniu wychodzi: 
    nowe_y = stare_y / (1 + 5 * krok).

In [10]:
def czesc_2_obliczenia():
    y_start = 1.0
    h = 0.5
    
    # (d) Jawny Euler
    # y1 = y0 + h * (-5 * y0)
    y_jawny = y_start + h * (-5 * y_start)
    
    # (f) Niejawny Euler
    # y1 = y0 / (1 + 5 * h)
    y_niejawny = y_start / (1 + 5 * h)
    
    # Dokładne rozwiązanie dla porównania
    y_dokladne = np.exp(-5 * 0.5)
    
    print(f"(d) Wynik jawnego Eulera dla t=0.5: {y_jawny}")
    print("Widzimy absurd! Wynik stał się ujemny. To dowód na niestabilność wykazaną w punkcie c.")
    print(f"(f) Wynik niejawnego Eulera dla t=0.5: {y_niejawny:.4f}")
    print(f"Dla porównania, wynik idealny (matematyczny) to: {y_dokladne:.4f}")

czesc_2_obliczenia()

(d) Wynik jawnego Eulera dla t=0.5: -1.5
Widzimy absurd! Wynik stał się ujemny. To dowód na niestabilność wykazaną w punkcie c.
(f) Wynik niejawnego Eulera dla t=0.5: 0.2857
Dla porównania, wynik idealny (matematyczny) to: 0.0821


### Część 3: Poszukiwanie optymalnego kroku 'h' (Podpunkt g)

**O co tu chodzi:** Wcześniej użyliśmy jawnego Eulera z ogromnym krokiem (0.5) i wybuchło nam to w twarz. Teraz odwracamy problem: prowadzimy dochodzenie, jak mały musi być krok 'h' (czyli na ile małych kawałków podzielić czas od 0 do 0.5), aby nasz numeryczny wynik różnił się od prawdy o mniej niż 0.001.

**Co robimy w tej komórce:** Uruchamiamy pętlę. Zaczynamy od podziału na 1 krok, potem 2, potem 3... Dla każdej liczby kroków liczymy ostateczny wynik jawnym Eulerem i sprawdzamy błąd w porównaniu do prawdziwego równania e^(-2.5). Gdy błąd spadnie poniżej progu 0.001, zatrzymujemy maszynę i wyświetlamy wynik.

In [11]:
def czesc_3_szukanie_kroku():
    t_koncowe = 0.5
    tolerancja = 0.001
    y_dokladne = np.exp(-5 * t_koncowe)
    
    liczba_krokow = 1
    
    while True:
        # Wyliczamy aktualny rozmiar kroku h
        h = t_koncowe / liczba_krokow
        
        # Ostateczna wartość z metody jawnego Eulera to (1 - 5h)^n
        y_numeryczne = (1 - 5 * h)**liczba_krokow
        
        # Sprawdzamy błąd bezwzględny
        blad = abs(y_numeryczne - y_dokladne)
        
        if blad < tolerancja:
            print(f"Osiągnięto wymaganą dokładność!")
            print(f"Minimalna liczba kroków: {liczba_krokow}")
            print(f"Maksymalny dopuszczalny krok h: {h:.6f}")
            print(f"Błąd wyniósł: {blad:.6f}")
            break
            
        liczba_krokow += 1

czesc_3_szukanie_kroku()

Osiągnięto wymaganą dokładność!
Minimalna liczba kroków: 257
Maksymalny dopuszczalny krok h: 0.001946
Błąd wyniósł: 0.000999


### Część 4: Wnętrze niejawnego Eulera (Podpunkt h)

**O co tu chodzi:** Niejawny Euler polega na tym, że nowe_y występuje po obu stronach równania. Dla prostych równań jak nasze (-5y), mogliśmy sobie to przekształcić ręcznie na kartce (jak w punkcie f). Ale w skomplikowanych problemach komputer nie potrafi tego tak łatwo wyciągnąć, więc musi... zgadywać i poprawiać swój błąd (iterować). 

* **Metoda bezpośredniej iteracji:** Bierzemy stare_y i wkładamy po prawej stronie wzoru Eulera z nadzieją, że wypluje lepsze nowe_y. Żeby to nie zwariowało, poprawki muszą być coraz mniejsze. Matematycznie, mnożnik w tej metodzie to wartość bezwzględna naszej stałej (-5) pomnożonej przez krok 'h'. Aby to zadziałało (zbiegało), mnożnik |5*h| musi być mniejszy od 1. Z tego wynika, że krok 'h' musi być mniejszy niż 1/5, czyli h < 0.2.
* **Czy metoda Newtona ma tu sens?** Metoda Newtona to matematyczny "walec". Dla równań liniowych (gdzie 'y' występuje po prostu jako y, a nie y do kwadratu czy w sinusie), metoda Newtona znajduje IDEALNE przybliżenie w dokładnie jednej iteracji. Dlatego odpowiedź brzmi: TAK, użycie jej byłoby w 100% uzasadnione i najszybsze.

In [12]:
def czesc_4_iteracje():
    # Pokazujemy tylko formalny wynik wyliczony w opisie
    max_h_iteracji = 1 / 5.0
    print(f"(h) Maksymalny krok dla zbieżności metody iteracji bezpośredniej to h < {max_h_iteracji}")
    print("Jeśli weźmiemy h równe 0.2 lub więcej, komputer wpadnie w nieskończoną pętlę i z każdym ")
    print("krokiem będzie pokazywał coraz większe bzdury zamiast zbliżać się do rozwiązania.")
    print("\nMetoda Newtona dla tego (liniowego) równania zbiegnie w zaledwie 1 kroku, ")
    print("jest to więc jak najbardziej uzasadnione podejście.")

czesc_4_iteracje()

(h) Maksymalny krok dla zbieżności metody iteracji bezpośredniej to h < 0.2
Jeśli weźmiemy h równe 0.2 lub więcej, komputer wpadnie w nieskończoną pętlę i z każdym 
krokiem będzie pokazywał coraz większe bzdury zamiast zbliżać się do rozwiązania.

Metoda Newtona dla tego (liniowego) równania zbiegnie w zaledwie 1 kroku, 
jest to więc jak najbardziej uzasadnione podejście.


### Zadanie 5: Stabilność układu równań

**O co tu chodzi:** W poprzednim zadaniu mieliśmy jedno równanie i jedną liczbę mówiącą o "tempie spadku" (-5). Tutaj mamy układ dwóch połączonych równań, które wpływają na siebie nawzajem. Żeby ocenić stabilność takiego systemu, musimy wyciągnąć z niego "wspólną esencję" – w algebrze liniowej odpowiadają za to tzw. wartości własne (eigenvalues) macierzy.

**Haczyk z liczbami zespolonymi:** Kiedy zbudujemy macierz z naszych równań i policzymy jej wartości własne, okaże się, że są to liczby zespolone (zawierają literkę 'i', a w Pythonie 'j'). Z perspektywy fizyki oznacza to, że nasz układ nie tylko dąży do wygaszenia (stabilizacji), ale po drodze ulega oscylacjom (trochę nim "buja"). Metoda Eulera słabo radzi sobie z oscylacjami. Warunek na jej stabilność mówi, że po przemnożeniu wartości własnej przez krok 'h' i dodaniu 1, cały wynik (jako wektor na płaszczyźnie zespolonej) musi zmieścić się w kole o promieniu 1.

**Co robimy w tej komórce:** 1. Zapisujemy współczynniki z zadania w formie macierzy $2 \times 2$.
2. Używamy `numpy` do znalezienia wartości własnych.
3. Wstawiamy otrzymaną wartość własną do warunku stabilności $|1 + h \cdot \lambda| \le 1$.
4. Używamy `sympy`, aby komputer sam ułożył z tego nierówność i wyliczył nam dokładny, bezpieczny zakres kroku 'h'.

In [13]:
import numpy as np
import sympy as sp

def zadanie_5_stabilnosc():
    # 1. Budujemy macierz układu z równań:
    # y1' = -2*y1 + 1*y2  -> wiersz [-2, 1]
    # y2' = -1*y1 - 2*y2  -> wiersz [-1, -2]
    A = np.array([
        [-2,  1],
        [-1, -2]
    ])
    
    # 2. Obliczamy wartości własne (eigenvalues)
    wartosci_wlasne = np.linalg.eigvals(A)
    print(f"Wartości własne macierzy: {wartosci_wlasne}")
    
    # Zauważ, że wyszły nam liczby zespolone: -2 + 1j oraz -2 - 1j.
    # Obie mają tę samą "wielkość", więc do warunku bierzemy jedną z nich.
    # W matematyce zapiszemy to jako lambda = -2 + i
    
    # 3. Definiujemy zmienną 'h' (krok musi być liczbą rzeczywistą dodatnią)
    h = sp.Symbol('h', real=True, positive=True)
    
    # Nasze wyrażenie to: 1 + h * lambda, czyli 1 + h * (-2 + i)
    # Wymnażając to, dostajemy: (1 - 2h) + i*h
    
    # Rozdzielamy to na część rzeczywistą i urojoną
    czesc_rzeczywista = 1 - 2 * h
    czesc_urojona = h
    
    # Warunek stabilności mówi, że moduł tej liczby musi być <= 1.
    # Dla ułatwienia sprawdzamy, czy moduł podniesiony do kwadratu jest <= 1^2
    # Moduł^2 = (Część Rzeczywista)^2 + (Część Urojona)^2
    modul_kwadrat = sp.expand(czesc_rzeczywista**2 + czesc_urojona**2)
    
    print(f"\nWarunek stabilności (Moduł^2 <= 1) przybiera postać:")
    print(f"{modul_kwadrat} <= 1")
    
    # 4. Rozwiązujemy nierówność
    nierownosc = modul_kwadrat <= 1
    rozwiazanie = sp.solve(nierownosc, h)
    
    print(f"\nOstateczne rozwiązanie:")
    print(f"Metoda Eulera jest stabilna dla kroku h należącego do przedziału:")
    print(rozwiazanie)

# Uruchamiamy obliczenia
zadanie_5_stabilnosc()

Wartości własne macierzy: [-2.+1.j -2.-1.j]

Warunek stabilności (Moduł^2 <= 1) przybiera postać:
5*h**2 - 4*h + 1 <= 1

Ostateczne rozwiązanie:
Metoda Eulera jest stabilna dla kroku h należącego do przedziału:
h <= 4/5


### Zadanie 6: Empiryczny rząd zbieżności metody Eulera

**O co tu chodzi:** W tym zadaniu dostajemy równanie, dla którego z góry znamy idealne, matematyczne rozwiązanie (jest to po prostu t do potęgi alfa). Dzięki temu możemy dokładnie zmierzyć, jak bardzo myli się nasza metoda numeryczna w każdym kroku.

**Po co w ogóle liczymy ten "rząd zbieżności"?** W teorii klasyczna metoda Eulera ma zbieżność rzędu 1. Oznacza to, że jeśli zmniejszymy krok `h` o połowę (np. z 0.2 na 0.1), to błąd też powinien spaść o połowę. 
Żeby jednak ta teoria działała, rozwiązanie musi być "odpowiednio gładkie" – konkretnie, jego druga pochodna nie może nigdzie wybuchać do nieskończoności.

**Wyjaśnienie wyników (czyli dlaczego to zadanie ma haczyk):**
Wzór na drugą pochodną naszego dokładnego rozwiązania to `alfa * (alfa - 1) * t do potęgi (alfa - 2)`.
Zauważ, co się dzieje dla małych alf (np. alfa = 1.5 albo 1.1) na samym starcie, gdzie t=0:
* Dla alfa = 2.5, potęga w drugiej pochodnej jest dodatnia (0.5). Wszystko jest stabilne i gładkie. Rząd zbieżności wyjdzie nam w okolicach 1.
* Dla alfa = 1.5, potęga robi się ujemna (-0.5). Druga pochodna w zerze dąży do nieskończoności (dzielenie przez zero)! 
* Dla alfa = 1.1 problem jest jeszcze silniejszy (potęga -0.9).

Skutek? Dla mniejszych alf funkcja na starcie załamuje się tak gwałtownie, że metoda Eulera całkowicie się gubi i nie nadąża z poprawkami. Zobaczysz w wynikach, że rząd zbieżności drastycznie spadnie poniżej teoretycznej jedynki (zbliży się do wartości samej alfy).

In [14]:
import numpy as np

def badanie_zbieznosci():
    alfy = [2.5, 1.5, 1.1]
    kroki_h = [0.2, 0.1, 0.05]
    t_max = 1.0  # Symulujemy rozwiązanie do t=1
    
    for alpha in alfy:
        print(f"\n{'='*40}")
        print(f"Badanie dla alfa = {alpha}")
        print(f"{'='*40}")
        
        bledy = []
        
        for h in kroki_h:
            # Tworzymy wektor czasu (węzły)
            # Dodajemy malutki margines h/10, żeby upewnić się, że t_max znajdzie się w tablicy
            t_values = np.arange(0, t_max + h/10, h)
            y_num = np.zeros_like(t_values)
            
            # Klasyczna metoda Eulera
            for i in range(len(t_values) - 1):
                t = t_values[i]
                
                # Zabezpieczenie dla t=0 (żeby numpy nie narzekał na 0 do ujemnej potęgi wewnątrz biblioteki)
                if t == 0:
                    pochodna = 0.0
                else:
                    pochodna = alpha * (t ** (alpha - 1))
                    
                y_num[i+1] = y_num[i] + h * pochodna
            
            # Obliczamy dokładne rozwiązanie w węzłach
            y_dokladne = t_values ** alpha
            
            # Liczymy maksymalny błąd numeryczny dla danego kroku h
            max_blad = np.max(np.abs(y_num - y_dokladne))
            bledy.append(max_blad)
            
            print(f"Krok h={h:.2f} | Maksymalny błąd: {max_blad:.5f}")
            
        print("-" * 40)
        print("Empiryczny Rząd Zbieżności (EOC):")
        # Liczymy EOC ze wzoru: log(blad1 / blad2) / log(h1 / h2)
        for i in range(len(kroki_h) - 1):
            h1, h2 = kroki_h[i], kroki_h[i+1]
            e1, e2 = bledy[i], bledy[i+1]
            
            eoc = np.log(e1 / e2) / np.log(h1 / h2)
            print(f"Przejście h={h1:.2f} -> h={h2:.2f}: EOC = {eoc:.4f}")

# Uruchomienie eksperymentu
badanie_zbieznosci()


Badanie dla alfa = 2.5
Krok h=0.20 | Maksymalny błąd: 0.23864
Krok h=0.10 | Maksymalny błąd: 0.12208
Krok h=0.05 | Maksymalny błąd: 0.06175
----------------------------------------
Empiryczny Rząd Zbieżności (EOC):
Przejście h=0.20 -> h=0.10: EOC = 0.9670
Przejście h=0.10 -> h=0.05: EOC = 0.9832

Badanie dla alfa = 1.5
Krok h=0.20 | Maksymalny błąd: 0.17539
Krok h=0.10 | Maksymalny błąd: 0.08424
Krok h=0.05 | Maksymalny błąd: 0.04083
----------------------------------------
Empiryczny Rząd Zbieżności (EOC):
Przejście h=0.20 -> h=0.10: EOC = 1.0581
Przejście h=0.10 -> h=0.05: EOC = 1.0448

Badanie dla alfa = 1.1
Krok h=0.20 | Maksymalny błąd: 0.18778
Krok h=0.10 | Maksymalny błąd: 0.09136
Krok h=0.05 | Maksymalny błąd: 0.04448
----------------------------------------
Empiryczny Rząd Zbieżności (EOC):
Przejście h=0.20 -> h=0.10: EOC = 1.0393
Przejście h=0.10 -> h=0.05: EOC = 1.0383
